In [ ]:
pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install odfpy pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd

# Chemin vers ton fichier feather
path_to_file = "/home/onyxia/stage_2A/data/classifications_core.feather"
path_to_file1="/home/onyxia/stage_2A/data/financials.feather"
path_to_file2="/home/onyxia/stage_2A/data/isin_codes.feather"
path_to_file3="/home/onyxia/stage_2A/data/legal_info.feather"
path_to_file4="/home/onyxia/stage_2A/data/names_1types.feather"


# Lecture du fichier
classif = pd.read_feather(path_to_file)
financials=pd.read_feather(path_to_file1)
isin_codes=pd.read_feather(path_to_file2)
legal_info=pd.read_feather(path_to_file3)
names=pd.read_feather(path_to_file4)

In [2]:


# 1. Forcer l'affichage de TOUTES les lignes (pas de coupure au milieu)
pd.set_option('display.max_rows', None)

# 2. Forcer l'affichage de TOUTES les colonnes
pd.set_option('display.max_columns', None)

# 3. Forcer l'affichage de tout le texte sans le tronquer (ex: les longs noms de pôles)
pd.set_option('display.max_colwidth', None)


on filtre les entreprises françaises

In [4]:
financials=financials[financials['country_code']=="FR"]
names=names[names['country_code']=="FR"]
legal_info=legal_info[legal_info['country_code']=="FR"]
isin_codes=isin_codes[isin_codes['country_code']=="FR"]
classif=classif[classif['country_code']=="FR"]

on supprime les variables avec des valeurs manquantes

In [3]:
# On crée une copie propre en supprimant les lignes où les variables financières reines sont NaN
financials_clean = financials.dropna(subset=['total_assets', 'sales', 'operating_revenue_turnover'], how='any')

# On vérifie la taille du nouveau dataset
print("Nouvelle taille du dataset :", len(financials_clean))
print("Pourcentage de lignes conservées :", (len(financials_clean) / len(financials)) * 100)

Nouvelle taille du dataset : 45197579
Pourcentage de lignes conservées : 40.56305996904639


In [ ]:

merge0 = pd.merge(
    financials_clean, 
    names, 
    on=['bvdid', 'country_code'],   
    how='left'      
)


In [ ]:
merge1 = pd.merge(
    merge0, 
    legal_info, 
    on=['bvdid','country_code'],
    how='left'      
)

In [ ]:
merge2 = pd.merge(
    merge1, 
    classif, 
    on=['bvdid','country_code'],
    how='left'      
)

In [ ]:
merge2.head()

,bvdid,country_code,consolidation_code,filing_type,closing_date,number_of_months,original_units,original_currency,exchange_rate_from_original_currency,total_assets,number_of_employees,operating_revenue_turnover,sales,name,entity_type,status,status_date,standardised_legal_form,national_legal_form,date_of_incorporation,type_of_entity,category_of_the_company,listed_delisted_unlisted,main_exchange,delisted_date,ipo_date,information_provider,bvd_major_sector,nace_rev_2_core_code_4_digits
0,00367EF,FR,C1,Annual report,19971231,12,thousands,EUR,1.0,6.106193e+09,23201.0,7.883901e+09,7.883901e+09,NaN,NaN,Inactive (no precision),201605.0,Public limited companies,Limited company - SA,183501.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,World'Vest Base Inc.,Other services,7311.0
1,00367EF,FR,C1,Annual report,19961231,12,thousands,EUR,1.0,5.992771e+09,22388.0,7.497900e+09,7.410089e+09,NaN,NaN,Inactive (no precision),201605.0,Public limited companies,Limited company - SA,183501.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,World'Vest Base Inc.,Other services,7311.0
2,00367EF,FR,C1,Annual report,19951231,12,thousands,EUR,1.0,5.201560e+09,20934.0,6.915087e+09,6.803190e+09,NaN,NaN,Inactive (no precision),201605.0,Public limited companies,Limited company - SA,183501.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,World'Vest Base Inc.,Other services,7311.0
3,00367EF,FR,C1,Annual report,19941231,12,thousands,EUR,1.0,4.356536e+09,18324.0,6.563692e+09,6.493413e+09,NaN,NaN,Inactive (no precision),201605.0,Public limited companies,Limited company - SA,183501.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,World'Vest Base Inc.,Other services,7311.0
4,00367EF,FR,C1,Annual report,19931231,12,thousands,EUR,1.0,4.061394e+09,18628.0,5.385871e+09,5.329160e+09,NaN,NaN,Inactive (no precision),201605.0,Public limited companies,Limited company - SA,183501.0,Corporate,Very large company,Unlisted,Unlisted,NaN,NaN,World'Vest Base Inc.,Other services,7311.0


In [11]:

# 1. On convertit la colonne au format Date en spécifiant le format d'origine '%Y%m%d'
# (%Y = année à 4 chiffres, %m = mois à 2 chiffres, %d = jour à 2 chiffres)
merge1['closing_datetime'] = pd.to_datetime(merge1['closing_date'], format='%Y%m%d', errors='coerce')
# 2. On extrait l'année très facilement
merge1['year'] = merge1['closing_datetime'].dt.year

# On regarde le résultat
print(merge1[['closing_date', 'year']].head())

   closing_date  year
0      20141231  2014
1      20131231  2013
2      20121231  2012
3      20111231  2011
4      20131231  2013


In [2]:
import pandas as pd

In [3]:
feder=pd.read_excel("/home/onyxia/stage_2A/data/base_projets_feder_fse_2007-2013.xlsx")

In [4]:
feder.columns

Index(['Nom de l'organisme bénéficiaire', 'Libellé du projet',
       'Coût total du projet (en euros)',
       'Montant du financement européen (en euros)', 'Fonds européen concerné',
       'Localisation du projet',
       'Nom de la commune de l'organisme bénéficiaire', 'Code département',
       'Année de programmation', 'Nom du programme européen',
       'Thématique du projet'],
      dtype='object')

In [5]:
# Le dictionnaire de correspondance (Ancien nom -> Nouveau nom propre)
dictionnaire_renommage = {
    "Nom de l'organisme bénéficiaire": "nom_beneficiaire",
    "Libellé du projet": "libelle_projet",
    "Coût total du projet (en euros)": "cout_total",
    "Montant du financement européen (en euros)": "montant_feder",
    "Fonds européen concerné": "fonds_europeen",
    "Localisation du projet": "localisation",
    "Nom de la commune de l'organisme bénéficiaire": "commune_beneficiaire",
    "Code département": "code_departement",
    "Année de programmation": "annee",
    "Nom du programme européen": "programme_europeen",
    "Thématique du projet": "thematique"
}

# On applique le changement sur ton DataFrame
feder = feder.rename(columns=dictionnaire_renommage)

# On vérifie le résultat
print(feder.columns)

Index(['nom_beneficiaire', 'libelle_projet', 'cout_total', 'montant_feder',
       'fonds_europeen', 'localisation', 'commune_beneficiaire',
       'code_departement', 'annee', 'programme_europeen', 'thematique'],
      dtype='object')


In [6]:

# 1. Liste des mots-clés à exclure (sensible à la casse ou converti en minuscules)
mots_cles_publics = [
    "mairie", "commune", "ville de",  "syndicat", "sivom", "ccas",
    "universite", "univ ", "cnrs", "inserm", "lycée", "college", "ecole",
    "association", "asso ", "chru", "chu ", "hopital", "centre hospitalier",
    "prefecture", "ministère", "region ", "département de", "quartier",
    "regional", "universitaire", "agence", "chambre", "centre", "conseil",
    "communaute", "municipal", "collectivite","conservatoire", "metropole",
    "communauté", "agglomération", "délégation", "université",
    "métropole", "région", "institut", "formation", "maison", "emploi",
    "public", "insertion", "office", "mission", "métropôle", "uni", 
    "region", "ligue", "fédération", "pôle", "réseau", "foyer", 
    "coopérative", "federation", "club", "espace", "snc", "solidarité", "académie", 
    "banque", "institu", "gaec", "hôpital", "musée", "les", "comite", "comité", "tourisme", 
    "c.h.u.", "enseignement", "amis", "reseau", "pays", "environnement", "groupement", "hopîtal", 
    "habitat", "résidence", "eplefpa", "fondation", "school", "ehpad", 
    "mutualité", "rectorat", "reseau", "amenagement", "developpement",
    "gaec", "commissariat", "cooperative", "direction", "habitat", "organisation", 
    "groupe", "européenne", "caisse", "jeunesse", "solidaire", "iut", "observatoire", 
    "service", "departemental", "lycee", "pole", "collège", "cercle", "sportive", "collectif",
    "coordination", "gites", "gîte","gîtes", "mission", "locale", "initiative", "crèches", 
    "aide", "école", "france", "territoire", "interdépartementale", 
    "hospitalier", "crédit", "communale", "communal", "port", "saint", 
    "planétarium", "préfecture", "prefecture","accompagnement", 
    "organisme", "francais", "nationale", "national", "parc", "paroisse"]
# On passe tout en minuscules pour ne pas rater de lignes
feder['nom_beneficiaire'] = feder['nom_beneficiaire'].str.lower()

# 2. Filtrage : on garde uniquement les lignes qui ne contiennent AUCUN mot-clé public
masque_entreprises = ~feder['nom_beneficiaire'].str.contains('|'.join(mots_cles_publics), na=False)
feder_entreprises = feder[masque_entreprises]

print(f"Après filtrage textuel, il reste {len(feder_entreprises)} bénéficiaires potentiels.")

Après filtrage textuel, il reste 28714 bénéficiaires potentiels.


In [7]:
feder_entreprises.head()

,nom_beneficiaire,libelle_projet,cout_total,montant_feder,fonds_europeen,localisation,commune_beneficiaire,code_departement,annee,programme_europeen,thematique
1,aerial,Mise en place de l'Unité Mixte Technologique ...,142000.0,69980.86,FEDER,ALSACE / BAS-RHIN / STRASBOURG-VILLE / STRASBO...,ILLKIRCH-GRAFFENSTADEN,67,2008,Programme Régional FEDER ALSACE,01.01.01~Activités de RDT dans les centres de ...
2,aerial,Mise en place de l'Unité Mixte Technologique ...,166325.0,49897.50,FEDER,ALSACE / BAS-RHIN / Plusieurs arrondissements ...,ILLKIRCH-GRAFFENSTADEN,67,2008,Programme Régional FEDER ALSACE,01.01.01~Activités de RDT dans les centres de ...
3,aerial,Mise en place de l'unité Mixte Technologique ...,200000.0,60000.00,FEDER,ALSACE / BAS-RHIN / Plusieurs arrondissements ...,ILLKIRCH-GRAFFENSTADEN,67,2009,Programme Régional FEDER ALSACE,01.01.01~Activités de RDT dans les centres de ...
4,aerial,Acquisition d'un spectromètre RMN,349000.0,139600.00,FEDER,ALSACE / BAS-RHIN / STRASBOURG-CAMPAGNE / ILLK...,ILLKIRCH-GRAFFENSTADEN,67,2013,Programme Régional FEDER ALSACE,01.01.01~Activités de RDT dans les centres de ...
27,agrivalor énergie,Création d'une unité de méthanisation sur l'ex...,6317857.0,500000.00,FEDER,ALSACE / HAUT-RHIN / RIBEAUVILLE / RIBEAUVILLE...,HIRSINGUE,68,2009,Programme Régional FEDER ALSACE,01.04.41~Énergies renouvelables : énergie biom...


In [13]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
import pandas as pd
import requests
from tqdm import tqdm

# ==========================================
# 1. FILTRAGE FEDER
# ==========================================
# (Remplace 'df' par le nom de ta variable si elle s'appelle autrement)
df_feder = feder_entreprises[feder_entreprises["fonds_europeen"] == "FEDER"].copy()
print(f"Nombre de lignes FEDER à traiter : {len(df_feder)}")


# ==========================================
# 2. FONCTION RECHERCHE API (SÉCURISÉE)
# ==========================================
def trouver_siren(nom, departement):
    if pd.isna(nom) or str(nom).strip() == "":
        return None

    url = "https://recherche-entreprises.api.gouv.fr/search"
    params = {"q": nom, "per_page": 1}

    # Formatage et intégration du département pour éviter les homonymes
    if pd.notna(departement):
        try:
            dept_str = str(int(float(departement))).zfill(2)
            params["departement"] = dept_str
        except ValueError:
            pass  # Si le département a un format étrange, on ne bloque pas

    try:
        # timeout=5 : si l'API met plus de 5s à répondre, on passe à la suite sans freezer
        reponse = requests.get(url, params=params, timeout=5)
        if reponse.status_code == 200:
            data = reponse.json()
            if data.get("results"):
                return data["results"][0].get("siren")
    except Exception:
        return None  # En cas de micro-coupure internet, on évite le plantage
    return None


# ==========================================
# 3. GESTION DU RESTART (CHECKPOINT)
# ==========================================
checkpoint_file = "feder_avec_siren_checkpoint.csv"

# Si tu as dû couper le code et que tu le relances, il reprend là où il s'est arrêté
if os.path.exists(checkpoint_file):
    print("Sauvegarde trouvée ! Reprise du travail en cours...")
    df_feder = pd.read_csv(checkpoint_file)
else:
    df_feder["siren"] = None


# ==========================================
# 4. BOUCLE DE TRAITEMENT ET SAUVEGARDE
# ==========================================
print("Lancement de la récupération des SIREN via l'API...")

compteur_sauvegarde = 0

for index, row in tqdm(df_feder.iterrows(), total=len(df_feder)):
    # Si le SIREN a déjà été trouvé lors d'un run précédent, on l'ignore
    if pd.notna(df_feder.at[index, "siren"]):
        continue

    # Appel de l'API
    siren = trouver_siren(row["nom_beneficiaire"], row["code_departement"])
    df_feder.at[index, "siren"] = siren

    compteur_sauvegarde += 1

    # Sauvegarde de secours toutes les 100 requêtes réussies
    if compteur_sauvegarde % 100 == 0:
        df_feder.to_csv(checkpoint_file, index=False)

# ==========================================
# 5. SAUVEGARDE FINALE
# ==========================================
df_feder.to_csv("/home/onyxia/stage_2A/data/feder_avec_siren_final.csv", index=False)

# On nettoie le fichier temporaire de checkpoint devenu inutile
if os.path.exists(checkpoint_file):
    os.remove(checkpoint_file)

print("\n Extraction terminée avec succès !")
print("Le fichier final est disponible sous le nom : 'feder_avec_siren_final.csv'")

# Affichage du taux de réussite pour ton rapport
taux_succes = df_feder["siren"].notna().mean() * 100
print(f"Pourcentage d'entreprises identifiées : {taux_succes:.2f}%")

Nombre de lignes FEDER à traiter : 11258
Sauvegarde trouvée ! Reprise du travail en cours...
Lancement de la récupération des SIREN via l'API...


  1%|          | 106/11258 [00:11<41:14,  4.51it/s]/tmp/ipykernel_55940/212665170.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '304614985' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_feder.at[index, "siren"] = siren
100%|██████████| 11258/11258 [52:18<00:00,  3.59it/s] 


 Extraction terminée avec succès !
Le fichier final est disponible sous le nom : 'feder_avec_siren_final.csv'
Pourcentage d'entreprises identifiées : 67.32%


In [5]:
feder_siren=pd.read_csv("/home/onyxia/stage_2A/data/feder_avec_siren_final.csv")

In [6]:

# 2. On filtre pour supprimer les lignes où le SIREN est vide (NaN)
feder_traitement = feder_siren[feder_siren['siren'].notna()].copy()

# 3. On force le SIREN à être au format texte propre (9 chiffres, sans le .0)
feder_traitement['siren'] = feder_traitement['siren'].astype(int).astype(str).str.zfill(9)

print(f"Nombre d'entreprises prêtes pour la fusion : {len(feder_traitement)}")

# 4. Sauvegarde de ta base d'entreprises traitées
feder_traitement.to_csv("/home/onyxia/stage_2A/data/feder_final_clean.csv", index=False)

Nombre d'entreprises prêtes pour la fusion : 7579


In [24]:
# 1. On check la longueur des SIREN du fichier FEDER
print("--- DISTRIBUTION DES LONGUEURS DANS FEDER ---")
# On force en string pour pouvoir compter les caractères
longueurs_feder = feder_traitement['siren'].astype(str).str.len()
print(longueurs_feder.value_counts())

# 2. On check la longueur des BvDID du fichier Orbis
print("\n--- DISTRIBUTION DES LONGUEURS DANS ORBIS ---")
longueurs_orbis = financials['bvdid'].astype(str).str.len()
print(longueurs_orbis.value_counts())

--- DISTRIBUTION DES LONGUEURS DANS FEDER ---
siren
9    7579
Name: count, dtype: int64

--- DISTRIBUTION DES LONGUEURS DANS ORBIS ---
bvdid
9     17597450
14        2228
7         1117
8           10
Name: count, dtype: int64


In [7]:

# 1. SÉCURITÉ : On force le type string sur Orbis pour manipuler le texte
financials["bvdid"] = financials["bvdid"].astype(str)

# 2. CORRECTION DES LONGUEURS


def corriger_bvdid(valeur):
    # Si c'est un SIRET (longueur 14), on ne garde que les 9 premiers chiffres
    if len(valeur) == 14:
        return valeur[:9]
    # Sinon, on garde la valeur telle quelle
    return valeur


# On applique la correction
financials["siren"] = financials["bvdid"].apply(corriger_bvdid)

# On remet les zéros au début pour tout le monde (longueurs 7, 8 -> 9)
financials["siren"] = financials["siren"].str.zfill(9)

# On fait pareil pour le FEDER par sécurité
feder_traitement["siren"] = feder_traitement["siren"].astype(str).str.zfill(9)

# =====================================================================
# 3. LA FUSION (INNER MERGE POUR LE PANEL)
# =====================================================================
# On utilise un 'inner' merge pour ne garder QUE les entreprises FEDER 
# mais en récupérant TOUTES leurs années d'historique financier disponibles dans Orbis.

df_panel_final = pd.merge(feder_traitement, financials, on="siren", how="inner")

print(f"Base FEDER de départ : {len(feder_traitement)} entreprises.")
print(f"Nombre de lignes dans ton panel final (Entreprises x Années) : {len(df_panel_final)}")
print(
    f"Nombre unique d'entreprises FEDER retrouvées dans Orbis : {df_panel_final['siren'].nunique()}"
)

Base FEDER de départ : 7579 entreprises.
Nombre de lignes dans ton panel final (Entreprises x Années) : 47169
Nombre unique d'entreprises FEDER retrouvées dans Orbis : 2483


In [8]:
merge0 = pd.merge(
    df_panel_final, 
    legal_info, 
    on=['bvdid', 'country_code'],   
    how='left'      
)


In [9]:
merge1 = pd.merge(
    merge0, 
    names, 
    on=['bvdid', 'country_code'],   
    how='left'      
)


In [12]:
# 1. On compte le nombre d'années UNIQUES pour chaque SIREN
annees_par_boite = merge1.groupby('siren')['year'].nunique()

# 2. On regarde la distribution (combien de boîtes ont 1 an, 2 ans, 5 ans, etc.)
print("--- DISTRIBUTION DU NOMBRE D'ANNÉES PAR ENTREPRISE ---")
print(annees_par_boite.value_counts().sort_index())

# 3. Quelques statistiques descriptives rapides
print(f"\nNombre moyen d'années disponibles : {annees_par_boite.mean():.1f} ans")
print(f"Médiane des années disponibles     : {annees_par_boite.median():.1f} ans")

--- DISTRIBUTION DU NOMBRE D'ANNÉES PAR ENTREPRISE ---
year
1      72
2      77
3      96
4      98
5     148
6     142
7     162
8     156
9     112
10     92
11     93
12     85
13     85
14     78
15     63
16     77
17     74
18     93
19     92
20    120
21    174
22    266
23     25
24      2
26      1
Name: count, dtype: int64

Nombre moyen d'années disponibles : 12.3 ans
Médiane des années disponibles     : 11.0 ans


In [14]:
import pandas as pd

# On s'assure que les colonnes temporelles sont bien lues comme des entiers
merge1["year"] = merge1["year"].astype(int)
merge1["annee"] = merge1["annee"].astype(int)

# =====================================================================
# ÉTAPE A : CRÉATION DES VARIABLES ÉCONOMÉTRIQUES
# =====================================================================
# Calcul du temps relatif (t - T_i)
merge1["temps_relatif"] = merge1["year"] - merge1["annee"]

# Création de la dummy 'post' (1 si l'année du bilan >= année de l'aide)
merge1["post"] = (merge1["year"] >= merge1["annee"]).astype(int)


# =====================================================================
# ÉTAPE B : DIAGNOSTIC DES DOUBLONS (MULTI-TRAITEMENTS)
# =====================================================================
print("=== 1. DIAGNOSTIC DES DOUBLONS (SIREN x ANNÉE) ===")

# On compte combien de lignes existent pour chaque couple (siren, year)
verif_doublons = merge1.groupby(["siren", "year"]).size()
nb_couples_dupliques = (verif_doublons > 1).sum()

print(f"Nombre de couples (siren, année) dupliqués : {nb_couples_dupliques}")

if nb_couples_dupliques > 0:
    print(
        "⚠️ ATTENTION : Certaines entreprises ont plusieurs lignes pour la même année."
    )
    print("Cela arrive si elles ont touché plusieurs subventions FEDER.")
else:
    print("✅ Parfait ! Aucune entreprise n'a de doublon pour une même année.")


# =====================================================================
# ÉTAPE C : DISTRIBUTION TEMPORELLE AUTOUR DU CHOC
# =====================================================================
print("\n=== 2. OBSERVATIONS DISPONIBLES AUTOUR DU CHOC (TEMPS RELATIF) ===")
# Zoom sur la fenêtre critique de l'Event Study (de -5 ans avant à +5 ans après)
zoom_temps = merge1[(merge1["temps_relatif"] >= -5) & (merge1["temps_relatif"] <= 5)]
print(zoom_temps["temps_relatif"].value_counts().sort_index())

print("\n=== 3. PROPORTION AVANT / APRÈS LE CHOC (VARIABLE POST) ===")
print(merge1["post"].value_counts(normalize=True) * 100)

=== 1. DIAGNOSTIC DES DOUBLONS (SIREN x ANNÉE) ===
Nombre de couples (siren, année) dupliqués : 7117
⚠️ ATTENTION : Certaines entreprises ont plusieurs lignes pour la même année.
Cela arrive si elles ont touché plusieurs subventions FEDER.

=== 2. OBSERVATIONS DISPONIBLES AUTOUR DU CHOC (TEMPS RELATIF) ===
temps_relatif
-5    2154
-4    2330
-3    2479
-2    2658
-1    2892
 0    3163
 1    3246
 2    3112
 3    2889
 4    2456
 5    1939
Name: count, dtype: int64

=== 3. PROPORTION AVANT / APRÈS LE CHOC (VARIABLE POST) ===
post
0    59.100257
1    40.899743
Name: proportion, dtype: float64


In [2]:
merge4=pd.read_csv("/home/onyxia/stage_2A/merge4.csv")

/tmp/ipykernel_7259/1256282770.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  merge4=pd.read_csv("/home/onyxia/stage_2A/merge4.csv")


In [3]:
annees_target = list(range(2002, 2019))  # 2002 à 2018
seuil_annees_minimal = 5

print("🧹 Étape 1 : Filtrage de la base merge4...")
merge4["siren_clean"] = merge4["bvdid"].astype(str).str.extract(r"(\d{9})")
merge4_propre = merge4.dropna(subset=["siren_clean"]).copy()

merge4_propre["year"] = merge4_propre["year"].astype(int)
df_periode = merge4_propre[merge4_propre["year"].isin(annees_target)]

# Filtrage présence minimale
comptage_annees = df_periode.groupby("siren_clean")["year"].nunique()
siren_valides_periode = comptage_annees[
    comptage_annees >= seuil_annees_minimal
].index.tolist()

merge4_filtre = df_periode[
    df_periode["siren_clean"].isin(siren_valides_periode)
].copy()

print(f"-> Nombre total de SIREN uniques cibles : {len(siren_valides_periode)}")

🧹 Étape 1 : Filtrage de la base merge4...
-> Nombre total de SIREN uniques cibles : 729494


In [ ]:
import os
import time
import pandas as pd
import requests

# =====================================================================
# ÉTAPE 1 : Nettoyage et harmonisation du format des SIREN
# =====================================================================
print("🧹 Étape 1 : Nettoyage et harmonisation du format des SIREN...")

# On force les SIREN de Colab à être du texte propre (9 chiffres, pas de .0)
merge4_filtre["siren_clean"] = (
    merge4_filtre["siren_clean"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(9)
)

# Extraction des SIREN uniques à traiter
siren_valides_periode = merge4_filtre["siren_clean"].unique().tolist()
print(
    f"-> Nombre total de SIREN uniques dans ta table merge4 : {len(siren_valides_periode)}"
)


# =====================================================================
# ÉTAPE 2 : Gestion de la reprise sur ton VRAI fichier à la racine
# =====================================================================
print("\n🔄 Étape 2 : Connexion au fichier de sauvegarde historique...")

# LE CHEMIN ABSOLU EXACT QUE TU AS DEVANT LES YEUX :
checkpoint_path = "/home/onyxia/stage_2A/api_geo_checkpoint.csv"

siren_deja_traites = set()

if os.path.exists(checkpoint_path) and os.path.getsize(checkpoint_path) > 0:
    try:
        # Lecture de ton gros fichier de 75 000+ lignes
        df_existant = pd.read_csv(checkpoint_path, dtype={"siren_clean": str})

        # Sécurité : on nettoie aussi les SIREN du CSV pour être sûr que la comparaison marche
        df_existant["siren_clean"] = (
            df_existant["siren_clean"]
            .astype(str)
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(9)
        )

        siren_deja_traites = set(df_existant["siren_clean"].dropna().tolist())
        print(
            f"   ✅ REPRISE RÉUSSIE ! {len(siren_deja_traites)} entreprises chargées depuis ton fichier racine."
        )
    except Exception as e:
        print(f"⚠️ Erreur lors de la lecture du CSV : {e}")
else:
    print(
        f"❌ Attention : Le fichier {checkpoint_path} est introuvable à cet emplacement."
    )

# Soustraction pour ne garder QUE ce qu'il reste à chercher
liste_a_traiter = [
    s for s in siren_valides_periode if s not in siren_deja_traites
]
print(
    f"🎯 Objectif réel restant pour cette nuit : {len(liste_a_traiter)} requêtes API."
)


# =====================================================================
# ÉTAPE 3 : Boucle d'interrogation API (Haute vitesse)
# =====================================================================
if len(liste_a_traiter) > 0:
    print(
        "\n⚡ Étape 3 : Appel de l'API à vitesse maximale autorisée... Mode nuit activé 🌙"
    )

    for idx, siren in enumerate(liste_a_traiter):
        url = f"https://recherche-entreprises.api.gouv.fr/search?q={siren}"

        # 🏎️ 0.15s de pause = ~6.6 requêtes/seconde
        time.sleep(0.15)

        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                data = response.json()
                if data.get("results"):
                    siege = data["results"][0].get("siege", {})

                    nouvelle_ligne = {
                        "siren_clean": siren,
                        "departement": siege.get("departement"),
                        "commune_insee": siege.get("code_commune"),
                    }

                    df_row = pd.DataFrame([nouvelle_ligne])
                    fichier_existe_deja = os.path.exists(checkpoint_path)

                    # On écrit DIRECTEMENT à la suite de ton gros fichier à la racine
                    df_row.to_csv(
                        checkpoint_path,
                        mode="a",
                        index=False,
                        header=not fichier_existe_deja,
                    )

            elif response.status_code == 429:
                print(
                    f"⚠️ Micro-freinage requis à l'index {idx}. Pause de 2 secondes..."
                )
                time.sleep(2)

        except Exception:
            pass

        # Affichage toutes les 50 lignes pour suivre l'avancement
        if (idx + 1) % 50 == 0 or (idx + 1) == len(liste_a_traiter):
            print(
                f"   🚀 Avancement : {idx + 1}/{len(liste_a_traiter)} nouvelles entreprises faites."
            )

print("\n🎉 Fin du requêtage API !")

🧹 Étape 1 : Nettoyage et harmonisation du format des SIREN...
-> Nombre total de SIREN uniques dans ta table merge4 : 729494

🔄 Étape 2 : Connexion au fichier de sauvegarde historique...
   ✅ REPRISE RÉUSSIE ! 156356 entreprises chargées depuis ton fichier racine.
🎯 Objectif réel restant pour cette nuit : 573138 requêtes API.

⚡ Étape 3 : Appel de l'API à vitesse maximale autorisée... Mode nuit activé 🌙
   🚀 Avancement : 50/573138 nouvelles entreprises faites.
   🚀 Avancement : 100/573138 nouvelles entreprises faites.
   🚀 Avancement : 150/573138 nouvelles entreprises faites.
   🚀 Avancement : 200/573138 nouvelles entreprises faites.
   🚀 Avancement : 250/573138 nouvelles entreprises faites.
   🚀 Avancement : 300/573138 nouvelles entreprises faites.
   🚀 Avancement : 350/573138 nouvelles entreprises faites.
   🚀 Avancement : 400/573138 nouvelles entreprises faites.
   🚀 Avancement : 450/573138 nouvelles entreprises faites.
   🚀 Avancement : 500/573138 nouvelles entreprises faites.
   🚀